# Publicar la aplicación en Hugging Face Spaces

Este cuaderno crea (o actualiza) un Space público de Hugging Face con la carpeta `app/` del repositorio y le configura las variables y los secretos. No guarda ninguna clave en archivos: las lee de los *Secretos* de Colab y las envía al panel del Space.

**Antes de ejecutar**, en *Secretos* (icono de llave) deben existir y tener acceso:

| Secreto | Valor |
|---|---|
| `HF_TOKEN` | Token de Hugging Face con permiso **Write** (huggingface.co → Settings → Access Tokens) |
| `GROQ_API_KEY` | Clave de Groq |
| `SUPABASE_URL` | Project URL de Supabase |
| `SUPABASE_PUBLISHABLE_KEY` | Clave publicable (`sb_publishable_...`) |
| `CORPUS_PROPIETARIO_ID` | UUID de la cuenta que cargó el corpus |


In [ ]:
#@title 1. Descargar el repositorio
!pip -q install -U huggingface_hub
!rm -rf buscador-conocimiento
!git clone -q https://github.com/AlejaMejia29/buscador-conocimiento
print("Listo.")

In [ ]:
#@title 2. Crear el Space y configurar variables y secretos
import shutil
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

NOMBRE_SPACE = "buscador-conocimiento" #@param {type:"string"}
CORPUS_ID = "buscador_conocimiento-29dccc23ac01" #@param {type:"string"}

api = HfApi(token=userdata.get("HF_TOKEN"))
usuario = api.whoami()["name"]
repo = f"{usuario}/{NOMBRE_SPACE}"
api.create_repo(repo, repo_type="space", space_sdk="gradio", exist_ok=True)

for nombre in ["GROQ_API_KEY", "SUPABASE_PUBLISHABLE_KEY"]:
    api.add_space_secret(repo, nombre, userdata.get(nombre))
variables = {
    "SUPABASE_URL": userdata.get("SUPABASE_URL"),
    "CORPUS_ID": CORPUS_ID,
    "CORPUS_PROPIETARIO_ID": userdata.get("CORPUS_PROPIETARIO_ID"),
    "EMBEDDINGS_REVISION": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
    "GROQ_MODEL": "openai/gpt-oss-20b",
    "GRADIO_SERVER_NAME": "0.0.0.0",
}
for nombre, valor in variables.items():
    api.add_space_variable(repo, nombre, valor)
print("Space configurado:", repo)

In [ ]:
#@title 3. Subir los archivos de la aplicación
origen = Path("buscador-conocimiento/app")
destino = Path("space"); shutil.rmtree(destino, ignore_errors=True); destino.mkdir()
for archivo in ["app.py", "sistema.py", "almacenamiento.py", "servidor_mcp.py", "requirements.txt"]:
    shutil.copy(origen / archivo, destino / archivo)
shutil.copy(origen / "despliegue_hf" / "README.md", destino / "README.md")
api.upload_folder(folder_path=destino, repo_id=repo, repo_type="space",
                  commit_message="Publicar la aplicación desde el repositorio")
print("Página del Space:", f"https://huggingface.co/spaces/{repo}")
print("Aplicación:", f"https://{usuario.lower()}-{NOMBRE_SPACE.lower()}.hf.space")
print("La primera construcción tarda de 5 a 15 minutos. Espera el estado «Running».")